# LeetCode 1447 — Simplified Fractions bằng Python

Link đề: https://leetcode.com/problems/simplified-fractions/ · Độ khó: **Medium** · Chủ đề: Math, String, Number Theory, Euclidean Algorithm, Greatest Common Divisor

Skeleton LeetCode:

```python
class Solution:
    def simplifiedFractions(self, n: int) -> list[str]:
        ...
```

## 1. Mục tiêu học

1. Hiểu **phân số tối giản**: `a/b` tối giản ⇔ `gcd(a, b) = 1` (tử và mẫu **nguyên tố cùng nhau**).
2. Biết thuật toán **Euclid** tính ƯCLN và dùng `math.gcd` trong Python.
3. Dùng `fractions.Fraction` để so sánh/khử trùng phân số **chính xác** (không dùng số thực).
4. (Nâng cao) Sinh phân số tối giản **trực tiếp, đã sắp xếp** bằng **cây Stern–Brocot** và **dãy Farey**, không cần kiểm tra gcd.
5. Kiểm tra số lượng kết quả bằng hàm **Euler φ**: tổng `φ(2) + … + φ(n)`; tránh lỗi biên `range`, lỗi "chỉ kiểm tra `b % a`".

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Trả về danh sách **tất cả phân số tối giản** nằm **trong khoảng (0, 1)** (không lấy 0 và 1) có **mẫu số ≤ n**, mỗi phân số viết dạng chuỗi `"tử/mẫu"`. Thứ tự tuỳ ý.

**Ví dụ 1:** `n = 2` → `["1/2"]`.

**Ví dụ 2:** `n = 3` → `["1/2","1/3","2/3"]`.

**Ví dụ 3:** `n = 4` → `["1/2","1/3","1/4","2/3","3/4"]`. `"2/4"` không tối giản vì rút gọn được thành `"1/2"`.

**Ràng buộc:** `1 <= n <= 100`.

Vì thứ tự tuỳ ý, ta so sánh kết quả **sau khi sắp xếp** (hoặc so dạng `set`).

In [1]:
import math
import random
import time
from fractions import Fraction
from typing import List

examples = [(2, ["1/2"]), (3, ["1/2", "1/3", "2/3"]), (4, ["1/2", "1/3", "1/4", "2/3", "3/4"])]
for n, want in examples:
    print(f"n={n} -> {want}")

def same(a, b):
    # so sánh không phụ thuộc thứ tự, đồng thời phát hiện phần tử trùng
    return sorted(a) == sorted(b)

n=2 -> ['1/2']
n=3 -> ['1/2', '1/3', '2/3']
n=4 -> ['1/2', '1/3', '1/4', '2/3', '3/4']


## 3. Kiến thức nền

### 3.1 Ước chung lớn nhất (ƯCLN, gcd) và thuật toán Euclid

`gcd(a, b)` là số lớn nhất chia hết cả `a` và `b`. Euclid: `gcd(a, b) = gcd(b, a mod b)`, dừng khi số thứ hai bằng 0. Mỗi 2 bước số giảm ít nhất một nửa ⇒ `O(log min(a, b))`.

In [2]:
def gcd_euclid(a, b):
    while b:
        a, b = b, a % b
    return a

for a, b in [(12, 18), (4, 6), (3, 8), (100, 75)]:
    print(f"gcd({a},{b}) = {gcd_euclid(a, b)} | math.gcd = {math.gcd(a, b)}")

gcd(12,18) = 6 | math.gcd = 6
gcd(4,6) = 2 | math.gcd = 2
gcd(3,8) = 1 | math.gcd = 1
gcd(100,75) = 25 | math.gcd = 25


### 3.2 Phân số tối giản ⇔ `gcd(tử, mẫu) = 1`

Nếu `g = gcd(a, b) > 1` thì chia cả tử và mẫu cho `g` được phân số bằng nó với mẫu nhỏ hơn ⇒ chưa tối giản. Mỗi giá trị trong (0,1) có **đúng một** cách viết tối giản.

In [3]:
for a, b in [(2, 4), (4, 6), (3, 4), (6, 9)]:
    g = math.gcd(a, b)
    print(f"{a}/{b}: gcd={g} -> {'tối giản' if g == 1 else f'rút gọn thành {a//g}/{b//g}'}")

2/4: gcd=2 -> rút gọn thành 1/2
4/6: gcd=2 -> rút gọn thành 2/3
3/4: gcd=1 -> tối giản
6/9: gcd=3 -> rút gọn thành 2/3


### 3.3 `fractions.Fraction` — phân số chính xác

`Fraction(a, b)` tự rút gọn, so sánh và băm (hash) theo **giá trị** ⇒ cho vào `set` sẽ tự khử trùng. `numerator` / `denominator` là tử / mẫu đã tối giản.

In [4]:
f = Fraction(2, 4)
print(f, f.numerator, f.denominator)
print(Fraction(2, 4) == Fraction(1, 2), len({Fraction(1, 2), Fraction(2, 4), Fraction(3, 6)}))
print("Fraction(1, 3) < Fraction(2, 5):", Fraction(1, 3) < Fraction(2, 5))

1/2 1 2
True 1
Fraction(1, 3) < Fraction(2, 5): True


### 3.4 f-string để tạo chuỗi `"a/b"`

`f"{a}/{b}"` ghép số thành chuỗi đúng định dạng đề yêu cầu (không có khoảng trắng).

In [5]:
a, b = 3, 7
print(f"{a}/{b}", repr(f"{a}/{b}"), "| sai định dạng:", repr(f"{a} / {b}"), repr(str(a / b)))

3/7 '3/7' | sai định dạng: '3 / 7' '0.42857142857142855'


### 3.5 Hàm Euler φ — đếm số tử hợp lệ với một mẫu

`φ(b)` = số `a` trong `1..b` nguyên tố cùng nhau với `b`. Với `b ≥ 2`, đó chính là số phân số tối giản mẫu `b` trong (0,1). Vậy độ dài đáp án = `φ(2) + φ(3) + … + φ(n)`. Có thể tính mọi `φ` bằng sàng.

In [6]:
def phi_sieve(n):
    phi = list(range(n + 1))
    for p in range(2, n + 1):
        if phi[p] == p:                    # p nguyên tố
            for m in range(p, n + 1, p):
                phi[m] -= phi[m] // p
    return phi

phi = phi_sieve(12)
print("φ(1..12) =", phi[1:])
print("số phân số với n=4:", sum(phi[2:5]))

φ(1..12) = [1, 1, 2, 2, 4, 2, 6, 4, 6, 4, 10, 4]
số phân số với n=4: 5


## 4. Ý tưởng nền tảng / trực giác

Liệt kê tay với `n = 6`, mỗi hàng một mẫu số `b`, giữ lại tử `a` (1 ≤ a < b) có `gcd(a, b) = 1`:

| b | các tử a | tối giản (gcd = 1) | bị loại |
|---|---|---|---|
| 2 | 1 | 1/2 | |
| 3 | 1,2 | 1/3, 2/3 | |
| 4 | 1,2,3 | 1/4, 3/4 | 2/4 (= 1/2) |
| 5 | 1..4 | 1/5 … 4/5 | |
| 6 | 1..5 | 1/6, 5/6 | 2/6, 3/6, 4/6 |

Tổng `1+2+2+4+2 = 11 = φ(2)+…+φ(6)`. Quy tắc rất đơn giản: **duyệt mọi cặp `a < b ≤ n`, giữ lại cặp có gcd = 1**. Mỗi giá trị chỉ được giữ đúng một lần — lần mà nó ở dạng tối giản.

In [7]:
n = 6
for b in range(2, n + 1):
    keep = [f"{a}/{b}" for a in range(1, b) if math.gcd(a, b) == 1]
    drop = [f"{a}/{b}" for a in range(1, b) if math.gcd(a, b) > 1]
    print(f"b={b}: giữ {keep} | loại {drop}")

b=2: giữ ['1/2'] | loại []
b=3: giữ ['1/3', '2/3'] | loại []
b=4: giữ ['1/4', '3/4'] | loại ['2/4']
b=5: giữ ['1/5', '2/5', '3/5', '4/5'] | loại []
b=6: giữ ['1/6', '5/6'] | loại ['2/6', '3/6', '4/6']


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** chưa biết `gcd`, ta kiểm tra "tối giản" bằng định nghĩa: **không có** số `d` từ `2` đến `a` chia hết cả `a` và `b`.

In [8]:
def simplified_brute(n: int) -> List[str]:
    res = []
    for b in range(2, n + 1):
        for a in range(1, b):
            if all(not (a % d == 0 and b % d == 0) for d in range(2, a + 1)):
                res.append(f"{a}/{b}")
    return res

for n, want in examples:
    print(sorted(simplified_brute(n)), same(simplified_brute(n), want))

['1/2'] True
['1/2', '1/3', '2/3'] True
['1/2', '1/3', '1/4', '2/3', '3/4'] True


**Độ phức tạp:** ba vòng lồng nhau ⇒ `O(n³)` (với `n = 100` khoảng 1,6·10⁵ phép chia — vẫn chạy nhanh vì `n` nhỏ). Space `O(n²)` cho kết quả (khoảng `3n²/π² ≈ 3000` phân số khi `n = 100`).

**Nút thắt:** vòng kiểm tra ước chung có thể thay bằng một lần gọi `gcd` `O(log n)`.

In [9]:
t = time.perf_counter()
r = simplified_brute(100)
print(len(r), f"phân số, brute mất {time.perf_counter()-t:.3f}s")

3043 phân số, brute mất 0.006s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — gcd:** "không có ước chung ≥ 2" ⇔ `gcd(a, b) = 1`. Thay vòng kiểm tra bằng `math.gcd` ⇒ `O(n² log n)`.

**Insight 2 — khử trùng theo giá trị:** thay vì kiểm tra tối giản, sinh **mọi** `a/b`, đưa về `Fraction` (tự rút gọn) và cho vào `set`. Mỗi giá trị còn đúng một bản, ở dạng tối giản.

**Insight 3 — sinh trực tiếp (Stern–Brocot / Farey):** mọi phân số tối giản trong (0,1) xuất hiện **đúng một lần** khi liên tục lấy **trung bình cộng kiểu "mediant"** `(a+c)/(b+d)` giữa hai phân số kề nhau, bắt đầu từ `0/1` và `1/1`. Mediant của hai phân số kề nhau luôn tối giản (vì `bc − ad = 1`). Giới hạn mẫu ≤ n cho ra **dãy Farey `F_n`** đã sắp xếp.

**Quy tắc "nếu… thì…":** nếu `b + d > n` thì không sinh mediant giữa `a/b` và `c/d` nữa (mọi phân số nằm giữa chúng đều có mẫu ≥ `b + d`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt mọi cặp + `math.gcd`

**Giải thích:** hai vòng `b = 2..n`, `a = 1..b−1`; giữ khi `gcd(a, b) == 1`. Viết gọn bằng list comprehension.

In [10]:
def simplified_gcd(n: int) -> List[str]:
    return [f"{a}/{b}" for b in range(2, n + 1) for a in range(1, b) if math.gcd(a, b) == 1]

for n, want in examples:
    print(simplified_gcd(n), same(simplified_gcd(n), want))

['1/2'] True
['1/2', '1/3', '2/3'] True
['1/2', '1/3', '2/3', '1/4', '3/4'] True


**Code nộp LeetCode (cách 2)** — xem mục 9 (đây là lời giải chính). **Độ phức tạp:** Time `O(n² log n)`, Space `O(n²)` cho output.

### 7.2 Cách 3 — `set` các `Fraction` (khử trùng theo giá trị)

**Giải thích từng đoạn:**
1. Sinh mọi `Fraction(a, b)` với `1 ≤ a < b ≤ n` — `Fraction` tự rút gọn `2/4 → 1/2`.
2. `set` loại các giá trị trùng.
3. Đổi mỗi phân số về chuỗi bằng `numerator` và `denominator`.

In [11]:
def simplified_fraction_set(n: int) -> List[str]:
    vals = {Fraction(a, b) for b in range(2, n + 1) for a in range(1, b)}
    return [f"{f.numerator}/{f.denominator}" for f in vals]

for n, want in examples:
    print(sorted(simplified_fraction_set(n)), same(simplified_fraction_set(n), want))

['1/2'] True
['1/2', '1/3', '2/3'] True
['1/2', '1/3', '1/4', '2/3', '3/4'] True


**Code nộp LeetCode (cách 3):**

In [12]:
class SolutionFractionSet:
    def simplifiedFractions(self, n: int) -> List[str]:
        vals = {Fraction(a, b) for b in range(2, n + 1) for a in range(1, b)}
        return [f"{f.numerator}/{f.denominator}" for f in vals]

print(sorted(SolutionFractionSet().simplifiedFractions(4)))

['1/2', '1/3', '1/4', '2/3', '3/4']


**Độ phức tạp:** Time `O(n² log n)` nhưng hằng số lớn (mỗi `Fraction` tự tính gcd và là đối tượng Python), Space `O(n²)`. Ưu điểm: không cần biết gì về gcd, chỉ cần "so sánh giá trị chính xác".

### 7.3 Cách 4 — Cây Stern–Brocot (đệ quy mediant)

**Giải thích từng đoạn:**
1. Bắt đầu với hai "biên" `0/1` và `1/1`.
2. `gen(a, b, c, d)`: mediant `m = (a+c)/(b+d)`. Nếu `b + d > n` thì dừng.
3. Duyệt **trung thứ tự**: sinh nửa trái `gen(a, b, a+c, b+d)`, ghi `m`, rồi sinh nửa phải `gen(a+c, b+d, c, d)` ⇒ kết quả **tăng dần**.

Độ sâu đệ quy ≤ `n` (mỗi tầng mẫu tăng ít nhất 1), với `n ≤ 100` an toàn.

In [13]:
def simplified_stern_brocot(n: int) -> List[str]:
    res = []

    def gen(a, b, c, d):
        if b + d > n:
            return
        gen(a, b, a + c, b + d)
        res.append(f"{a + c}/{b + d}")
        gen(a + c, b + d, c, d)

    gen(0, 1, 1, 1)
    return res

for n, want in examples:
    print(simplified_stern_brocot(n), same(simplified_stern_brocot(n), want))

['1/2'] True
['1/3', '1/2', '2/3'] True
['1/4', '1/3', '1/2', '2/3', '3/4'] True


**Code nộp LeetCode (cách 4):**

In [14]:
class SolutionSternBrocot:
    def simplifiedFractions(self, n: int) -> List[str]:
        res = []

        def gen(a, b, c, d):
            if b + d <= n:
                gen(a, b, a + c, b + d)
                res.append(f"{a + c}/{b + d}")
                gen(a + c, b + d, c, d)

        gen(0, 1, 1, 1)
        return res

print(SolutionSternBrocot().simplifiedFractions(5))

['1/5', '1/4', '1/3', '2/5', '1/2', '3/5', '2/3', '3/4', '4/5']


**Độ phức tạp:** mỗi lời gọi hoặc sinh đúng 1 phân số, hoặc dừng ngay (số lời gọi dừng ≤ số phân số + 1) ⇒ Time `O(|kết quả|) = O(n²)`, **không cần gcd**. Space `O(n)` cho stack + output.

### 7.4 Cách 5 — Dãy Farey bằng công thức "số hạng kế tiếp" (lặp, không đệ quy)

Nếu `a/b < c/d` là hai số hạng kề nhau của `F_n`, số hạng kế tiếp là `(k·c − a)/(k·d − b)` với `k = ⌊(n + b)/d⌋`. Bắt đầu `0/1, 1/n`, lặp đến khi gặp `1/1`.

In [15]:
def simplified_farey(n: int) -> List[str]:
    res = []
    a, b, c, d = 0, 1, 1, n
    while not (c == 1 and d == 1):     # dừng khi tới 1/1 (n = 1 thì dừng ngay)
        res.append(f"{c}/{d}")
        k = (n + b) // d
        a, b, c, d = c, d, k * c - a, k * d - b
    return res

class SolutionFarey:
    def simplifiedFractions(self, n: int) -> List[str]:
        res, (a, b, c, d) = [], (0, 1, 1, n)
        while not (c == 1 and d == 1):
            res.append(f"{c}/{d}")
            k = (n + b) // d
            a, b, c, d = c, d, k * c - a, k * d - b
        return res

for n, want in examples:
    print(simplified_farey(n), same(SolutionFarey().simplifiedFractions(n), want))
print("n=1:", simplified_farey(1))

['1/2'] True
['1/3', '1/2', '2/3'] True
['1/4', '1/3', '1/2', '2/3', '3/4'] True
n=1: []


**Độ phức tạp:** Time `O(|kết quả|)`, Space `O(1)` ngoài output — tối ưu tuyệt đối vì phải in ra từng phân số.

So sánh tốc độ với `n = 100` (giới hạn đề) và `n = 1000` (ngoài đề, để thấy khác biệt):

In [16]:
for N in (100, 1000):
    for name, f in [("gcd", simplified_gcd), ("Fraction set", simplified_fraction_set),
                    ("Stern-Brocot", simplified_stern_brocot), ("Farey", simplified_farey)]:
        if N == 1000 and name in ("Fraction set", "Stern-Brocot"):
            continue   # Stern-Brocot sâu ~n tầng: n=1000 vượt giới hạn đệ quy mặc định
        t = time.perf_counter()
        r = f(N)
        print(f"n={N:4} {name:13} -> {len(r):6d} phân số ({time.perf_counter()-t:.4f}s)")

n= 100 gcd           ->   3043 phân số (0.0007s)
n= 100 Fraction set  ->   3043 phân số (0.0066s)
n= 100 Stern-Brocot  ->   3043 phân số (0.0006s)
n= 100 Farey         ->   3043 phân số (0.0005s)
n=1000 gcd           -> 304191 phân số (0.0767s)


n=1000 Farey         -> 304191 phân số (0.0733s)


## 8. TRACE từng bước

**Trace cách 2** với `n = 5`: in từng cặp `(a, b)`, gcd và quyết định.

| b | a | gcd | giữ? |
|---|---|---|---|
| 2 | 1 | 1 | ✔ 1/2 |
| 3 | 1, 2 | 1, 1 | ✔ 1/3, 2/3 |
| 4 | 1, 2, 3 | 1, **2**, 1 | ✔ 1/4, ✘ 2/4, ✔ 3/4 |
| 5 | 1..4 | 1 | ✔ cả 4 |

In [17]:
def trace_gcd(n):
    res = []
    for b in range(2, n + 1):
        for a in range(1, b):
            g = math.gcd(a, b)
            keep = g == 1
            if keep:
                res.append(f"{a}/{b}")
            print(f"b={b} a={a} gcd={g} -> {'giữ' if keep else 'loại'}")
    return res

print(trace_gcd(5))

b=2 a=1 gcd=1 -> giữ
b=3 a=1 gcd=1 -> giữ
b=3 a=2 gcd=1 -> giữ
b=4 a=1 gcd=1 -> giữ
b=4 a=2 gcd=2 -> loại
b=4 a=3 gcd=1 -> giữ
b=5 a=1 gcd=1 -> giữ
b=5 a=2 gcd=1 -> giữ
b=5 a=3 gcd=1 -> giữ
b=5 a=4 gcd=1 -> giữ
['1/2', '1/3', '2/3', '1/4', '3/4', '1/5', '2/5', '3/5', '4/5']


**Trace cách 5 (Farey)** với `n = 5`: biến `(a/b, c/d)` và hệ số `k` qua từng vòng.

In [18]:
def trace_farey(n):
    res = []
    a, b, c, d = 0, 1, 1, n
    step = 0
    while not (c == 1 and d == 1):
        res.append(f"{c}/{d}")
        k = (n + b) // d
        na, nb, nc, nd = c, d, k * c - a, k * d - b
        print(f"bước {step}: cặp kề ({a}/{b}, {c}/{d}) | k=({n}+{b})//{d}={k} -> kế tiếp {nc}/{nd}")
        a, b, c, d = na, nb, nc, nd
        step += 1
    return res

print(trace_farey(5))

bước 0: cặp kề (0/1, 1/5) | k=(5+1)//5=1 -> kế tiếp 1/4
bước 1: cặp kề (1/5, 1/4) | k=(5+5)//4=2 -> kế tiếp 1/3
bước 2: cặp kề (1/4, 1/3) | k=(5+4)//3=3 -> kế tiếp 2/5
bước 3: cặp kề (1/3, 2/5) | k=(5+3)//5=1 -> kế tiếp 1/2
bước 4: cặp kề (2/5, 1/2) | k=(5+5)//2=5 -> kế tiếp 3/5
bước 5: cặp kề (1/2, 3/5) | k=(5+2)//5=1 -> kế tiếp 2/3
bước 6: cặp kề (3/5, 2/3) | k=(5+5)//3=3 -> kế tiếp 3/4
bước 7: cặp kề (2/3, 3/4) | k=(5+3)//4=2 -> kế tiếp 4/5
bước 8: cặp kề (3/4, 4/5) | k=(5+4)//5=1 -> kế tiếp 1/1
['1/5', '1/4', '1/3', '2/5', '1/2', '3/5', '2/3', '3/4', '4/5']


## 9. Code nộp LeetCode

Lời giải chính (cách 2 — gcd), ngắn, rõ, đúng chữ ký `simplifiedFractions(self, n)`.

In [19]:
import math
from typing import List

class Solution:
    def simplifiedFractions(self, n: int) -> List[str]:
        return [f"{a}/{b}"
                for b in range(2, n + 1)
                for a in range(1, b)
                if math.gcd(a, b) == 1]

for n, want in examples:
    assert same(Solution().simplifiedFractions(n), want)
print("Solution: 3 ví dụ đề OK (so sau khi sort)")

Solution: 3 ví dụ đề OK (so sau khi sort)


## 10. Vì sao thuật toán đúng

**Cách 2:** mỗi giá trị `x ∈ (0,1)` có mẫu tối giản ≤ n có đúng một biểu diễn tối giản `p/q` (`gcd = 1`, `q ≤ n`, `1 ≤ p < q`). Vòng lặp duyệt mọi `1 ≤ a < b ≤ n`, nên gặp `(p, q)` đúng một lần và giữ nó; mọi cặp `(a, b)` khác có cùng giá trị đều có `gcd > 1` nên bị loại. Vậy kết quả = mỗi giá trị đúng một lần, ở dạng tối giản.

**Cách 3:** `Fraction` biểu diễn chuẩn hoá (tối giản), hai `Fraction` bằng nhau ⇔ cùng giá trị ⇒ `set` giữ mỗi giá trị một lần; `numerator/denominator` là dạng tối giản.

**Cách 4/5 (Stern–Brocot/Farey):** tính chất cổ điển: nếu `a/b < c/d` thoả `bc − ad = 1` thì mediant `(a+c)/(b+d)` nằm giữa, tối giản, cùng thoả tính chất với mỗi biên, và **mọi** phân số nằm giữa `a/b` và `c/d` đều có mẫu ≥ `b + d`. Bắt đầu từ `0/1, 1/1` (`1·1 − 0·1 = 1`), quy nạp cho thấy mọi phân số tối giản có mẫu ≤ n được sinh đúng một lần, theo thứ tự tăng dần. Công thức số hạng kế tiếp của Farey là hệ quả: `c/d` kế tiếp `a/b` là phân số có mẫu lớn nhất ≤ n thoả `b·c' − a·d' = 1`.

## 11. Phân tích Time / Space

Gọi `n` là giới hạn mẫu; số phân số kết quả `K = Σ_{b=2..n} φ(b) ≈ 3n²/π²` (≈ 3043 khi `n = 100`).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Brute thử ước | `O(n³)` | `O(1)` | định nghĩa trực tiếp |
| 2. gcd | `O(n² log n)` | `O(1)` | lời giải chuẩn |
| 3. Set Fraction | `O(n² log n)` (hằng số lớn) | `O(n²)` | khử trùng theo giá trị |
| 4. Stern–Brocot | `O(K)` | `O(n)` stack | sinh trực tiếp, đã sắp xếp |
| 5. Farey lặp | `O(K)` | `O(1)` | tối ưu, đã sắp xếp |

Mọi cách đều cần `O(K) = Θ(n²)` bộ nhớ cho chính output.

## 12. Bộ test

Ví dụ đề (so sau khi sort), edge case `n = 1`, kiểm tra **số lượng** bằng Euler φ, **không trùng**, **mọi phần tử tối giản và nằm trong (0,1)**, và các cách khớp nhau với mọi `n = 1..100`.

In [20]:
ALL = {
    "brute": simplified_brute,
    "gcd": simplified_gcd,
    "fraction_set": simplified_fraction_set,
    "stern_brocot": simplified_stern_brocot,
    "farey": simplified_farey,
    "SolutionFractionSet": lambda n: SolutionFractionSet().simplifiedFractions(n),
    "SolutionSternBrocot": lambda n: SolutionSternBrocot().simplifiedFractions(n),
    "SolutionFarey": lambda n: SolutionFarey().simplifiedFractions(n),
    "Solution": lambda n: Solution().simplifiedFractions(n),
}

def is_valid_answer(n, res):
    if len(res) != len(set(res)):
        return False                          # có phần tử trùng
    for s in res:
        a, b = map(int, s.split("/"))
        if s != f"{a}/{b}" or not (1 <= a < b <= n) or math.gcd(a, b) != 1:
            return False
    return len(res) == sum(phi_sieve(n)[2:])  # đủ số lượng

for n, want in examples + [(1, [])]:
    for name, f in ALL.items():
        assert same(f(n), want), (name, n)
print("Ví dụ đề + n=1: OK")

Ví dụ đề + n=1: OK


In [21]:
PHI = phi_sieve(100)
ref = {n: sorted(simplified_gcd(n)) for n in range(1, 101)}
for n in range(1, 101):
    for name, f in ALL.items():
        if name == "brute" and n > 60:
            continue
        out = f(n)
        assert sorted(out) == ref[n], (name, n)
    assert is_valid_answer(n, ref[n])
    assert len(ref[n]) == sum(PHI[2:n + 1])
print("n = 1..100: mọi cách khớp nhau, hợp lệ, đúng số lượng Σφ")
print("n=100 có", len(ref[100]), "phân số; 3n²/π² ≈", round(3 * 100**2 / math.pi**2))

n = 1..100: mọi cách khớp nhau, hợp lệ, đúng số lượng Σφ
n=100 có 3043 phân số; 3n²/π² ≈ 3040


In [22]:
# Stern-Brocot và Farey cho kết quả đã sắp xếp tăng dần theo giá trị
for n in (5, 17, 100):
    sb, fa = simplified_stern_brocot(n), simplified_farey(n)
    vals = [Fraction(s) for s in fa]
    assert sb == fa and vals == sorted(vals)
print("Stern-Brocot == Farey và tăng dần:", simplified_farey(5))

Stern-Brocot == Farey và tăng dần: ['1/5', '1/4', '1/3', '2/5', '1/2', '3/5', '2/3', '3/4', '4/5']


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không rút gọn ⇒ trùng giá trị

Liệt kê mọi `a/b` mà không kiểm tra gcd ⇒ có cả `"1/2"` và `"2/4"`.

In [23]:
def wrong_no_reduce(n):
    return [f"{a}/{b}" for b in range(2, n + 1) for a in range(1, b)]

out = wrong_no_reduce(4)
print("Sai :", out, "| hợp lệ?", is_valid_answer(4, out))
print("Đúng:", Solution().simplifiedFractions(4))

Sai : ['1/2', '1/3', '2/3', '1/4', '2/4', '3/4'] | hợp lệ? False
Đúng: ['1/2', '1/3', '2/3', '1/4', '3/4']


**Sửa:** thêm điều kiện `math.gcd(a, b) == 1`.

### Lỗi 2 — Chỉ kiểm tra "`a` không chia hết `b`"

Tưởng rằng `a/b` rút gọn được chỉ khi `b % a == 0`. Sai cả hai chiều: `4/6` rút gọn được dù `6 % 4 ≠ 0`; còn `1/b` luôn bị loại vì `b % 1 == 0`.

In [24]:
def wrong_divides(n):
    return [f"{a}/{b}" for b in range(2, n + 1) for a in range(1, b) if b % a != 0]

out = wrong_divides(6)
print("Sai :", sorted(out))
print("Đúng:", sorted(Solution().simplifiedFractions(6)))
print("Sai có '4/6'?", "4/6" in out, "| Sai có '1/2'?", "1/2" in out)

Sai : ['2/3', '2/5', '3/4', '3/5', '4/5', '4/6', '5/6']
Đúng: ['1/2', '1/3', '1/4', '1/5', '1/6', '2/3', '2/5', '3/4', '3/5', '4/5', '5/6']
Sai có '4/6'? True | Sai có '1/2'? False


**Sửa:** điều kiện đúng là `gcd(a, b) == 1` (không có **bất kỳ** ước chung nào > 1).

### Lỗi 3 — Lệch biên `range`: thiếu mẫu `n` hoặc lấy cả `a = b`

`range(a, b)` không lấy `b`. Viết `range(2, n)` thì thiếu mẫu `n`; viết `range(1, b + 1)` cho tử thì thêm `"1/1"`.

In [25]:
def wrong_missing_n(n):
    return [f"{a}/{b}" for b in range(2, n) for a in range(1, b) if math.gcd(a, b) == 1]

def wrong_include_one(n):
    return [f"{a}/{b}" for b in range(1, n + 1) for a in range(1, b + 1) if math.gcd(a, b) == 1]

print("Thiếu mẫu n :", wrong_missing_n(3), "| Đúng:", Solution().simplifiedFractions(3))
print("Thêm '1/1'  :", wrong_include_one(2), "| Đúng:", Solution().simplifiedFractions(2))
print("n=1 bản sai :", wrong_include_one(1), "| Đúng:", Solution().simplifiedFractions(1))

Thiếu mẫu n : ['1/2'] | Đúng: ['1/2', '1/3', '2/3']
Thêm '1/1'  : ['1/1', '1/2'] | Đúng: ['1/2']
n=1 bản sai : ['1/1'] | Đúng: []


**Sửa:** mẫu `range(2, n + 1)`, tử `range(1, b)`.

### Lỗi 4 — Sai định dạng chuỗi

Trả về số thực `0.5`, tuple `(1, 2)` hay chuỗi có khoảng trắng `"1 / 2"` đều sai định dạng.

In [26]:
def wrong_format(n):
    return [f"{a} / {b}" for b in range(2, n + 1) for a in range(1, b) if math.gcd(a, b) == 1]

out = wrong_format(3)
print("Sai :", out, "| hợp lệ?", is_valid_answer(3, out))
print("Đúng:", Solution().simplifiedFractions(3))

Sai : ['1 / 2', '1 / 3', '2 / 3'] | hợp lệ? False
Đúng: ['1/2', '1/3', '2/3']


**Sửa:** đúng `f"{a}/{b}"`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tối giản", "nguyên tố cùng nhau", "không có ước chung"** ⇒ `math.gcd(a, b) == 1`.
- **Đếm** số cặp nguyên tố cùng nhau ⇒ hàm **Euler φ** (sàng `O(n log log n)`).
- **So sánh phân số chính xác** ⇒ `Fraction` hoặc nhân chéo `a·d` vs `b·c` — tránh số thực.
- **Sinh phân số theo thứ tự tăng dần** / tìm phân số "kề" ⇒ **Farey / Stern–Brocot**.
- Đề "thứ tự tuỳ ý" ⇒ khi test, **sort** rồi so; thêm kiểm tra không trùng.

## 15. Biến thể của bài

1. **Đếm** số phân số tối giản (không cần liệt kê): `Σ φ(b)` — giải được cho `n` lớn (bài tập 4).
2. **Liệt kê theo thứ tự tăng dần:** dãy Farey (bài tập 5).
3. **K-th Smallest Prime Fraction** (LeetCode 786): phân số từ một mảng cho trước, dùng heap hoặc tìm nhị phân trên giá trị.
4. **Fraction Addition and Subtraction** (LeetCode 592): cộng trừ phân số, rút gọn bằng gcd.
5. **Phân số gần nhất với một số thực** có mẫu ≤ n: đi xuống cây Stern–Brocot.

In [27]:
def closest_fraction(x, n):
    # đi xuống cây Stern-Brocot để tìm phân số mẫu <= n gần x nhất (x trong (0,1))
    a, b, c, d = 0, 1, 1, 1
    best = min((Fraction(0, 1), Fraction(1, 1)), key=lambda f: abs(f - x))
    while b + d <= n:
        m = Fraction(a + c, b + d)
        if abs(m - x) < abs(best - x):
            best = m
        if m < x:
            a, b = a + c, b + d
        else:
            c, d = a + c, b + d
    return best

print(closest_fraction(Fraction(314159, 1000000), 10), closest_fraction(Fraction(1, 3), 100))

3/10 1/3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Thử mọi ước `d ≤ a` | `O(n³)` | `O(1)` | Khi chưa biết gcd, kiểm chứng |
| 2. gcd | Giữ cặp `gcd = 1` | `O(n² log n)` | `O(1)` | Lời giải chuẩn, phỏng vấn |
| 3. Set Fraction | Khử trùng theo giá trị | `O(n² log n)` | `O(n²)` | Khi muốn "so giá trị" an toàn |
| 4. Stern–Brocot | Đệ quy mediant | `O(K)` | `O(n)` | Cần thứ tự tăng dần, nâng cao |
| 5. Farey lặp | Công thức số hạng kế tiếp | `O(K)` | `O(1)` | Tối ưu nhất, nâng cao |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `len(simplifiedFractions(6))` bằng bao nhiêu?

**Bài 2 (tính tay):** liệt kê các phân số tối giản có mẫu **đúng bằng** 12.

**Bài 3 (điền TODO):** hoàn thiện `gcd_todo(a, b)` theo thuật toán Euclid (không dùng `math.gcd`).

**Bài 4 (tự viết lại):** viết `count_simplified(n)` trả về số phân số tối giản (đúng bằng `len(simplifiedFractions(n))`) mà **không** sinh danh sách, chạy nhanh với `n = 10^5`.

**Bài 5 (mini challenge):** viết `kth_fraction(n, k)` trả về chuỗi phân số tối giản **nhỏ thứ k** (1-indexed) trong (0,1) với mẫu ≤ n, không dùng số thực.

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def gcd_todo(a, b):
    while b:
        # TODO: thay (a, b) bằng (b, a mod b)
        pass
    return a

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Đếm theo từng mẫu. (2) Đó là `φ(2)+…+φ(6)`. (3) `1+2+2+4+2`.

**Bài 2:** (1) Tử `a` từ 1 đến 11. (2) Loại `a` chia hết cho 2 hoặc 3 (ước nguyên tố của 12). (3) Còn 1, 5, 7, 11.

**Bài 3:** (1) Gán đồng thời bằng tuple. (2) `a, b = b, a % b`. (3) Khi `b = 0` thì `a` là gcd.

**Bài 4:** (1) Dùng sàng `phi_sieve`. (2) Cộng `phi[2..n]`. (3) Sàng `O(n log log n)`, rất nhanh với `10^5`.

**Bài 5:** (1) Farey cho dãy đã sắp xếp. (2) Lặp công thức số hạng kế tiếp `k − 1` lần. (3) Hoặc sort danh sách `Fraction` — đúng nhưng chậm hơn; dùng để kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
ans1 = len(Solution().simplifiedFractions(6))
print("Bài 1:", ans1)
assert ans1 == 11

# Bài 2
ans2 = [f"{a}/12" for a in range(1, 12) if math.gcd(a, 12) == 1]
print("Bài 2:", ans2)
assert ans2 == ["1/12", "5/12", "7/12", "11/12"]

# Bài 3
def gcd_todo_sol(a, b):
    while b:
        a, b = b, a % b
    return a

rng = random.Random(3)
for _ in range(1000):
    a, b = rng.randint(0, 10**6), rng.randint(0, 10**6)
    assert gcd_todo_sol(a, b) == math.gcd(a, b)
print("Bài 3: OK")

Bài 1: 11
Bài 2: ['1/12', '5/12', '7/12', '11/12']
Bài 3: OK


In [30]:
# Bài 4
def count_simplified(n):
    return sum(phi_sieve(n)[2:]) if n >= 2 else 0

for n in range(1, 101):
    assert count_simplified(n) == len(Solution().simplifiedFractions(n))
t = time.perf_counter()
print("Bài 4: count_simplified(10^5) =", count_simplified(10**5), f"({time.perf_counter()-t:.2f}s)")

# Bài 5
def kth_fraction(n, k):
    a, b, c, d = 0, 1, 1, n
    for _ in range(k - 1):
        kk = (n + b) // d
        a, b, c, d = c, d, kk * c - a, kk * d - b
    return f"{c}/{d}"

for n in (2, 7, 30):
    ordered = sorted(Fraction(s) for s in Solution().simplifiedFractions(n))
    for k in range(1, len(ordered) + 1):
        f = ordered[k - 1]
        assert kth_fraction(n, k) == f"{f.numerator}/{f.denominator}"
print("Bài 5: kth_fraction(5, 4) =", kth_fraction(5, 4), "| OK")

Bài 4: count_simplified(10^5) = 3039650753 (0.03s)
Bài 5: kth_fraction(5, 4) = 2/5 | OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template gcd:**

```python
return [f"{a}/{b}" for b in range(2, n + 1) for a in range(1, b) if gcd(a, b) == 1]
```

**Template Farey (tăng dần, không gcd):**

```python
a, b, c, d = 0, 1, 1, n
while not (c == 1 and d == 1):
    res.append(f"{c}/{d}")
    k = (n + b) // d
    a, b, c, d = c, d, k * c - a, k * d - b
```

**Checklist:**
- [ ] Mẫu `2..n` (có `n`), tử `1..b−1` (không có 0 và `b`).
- [ ] Điều kiện `gcd(a, b) == 1`, không phải `b % a != 0`.
- [ ] Định dạng chuỗi đúng `"a/b"`, không khoảng trắng, không số thực.
- [ ] `n = 1` trả về `[]`.
- [ ] Số lượng khớp `Σ φ(b)`; không có phần tử trùng.